In [ ]:
!pip install pandas plotly folium matplotlib seaborn --quiet
print('All packages installed.')

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import folium
from folium.plugins import HeatMap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, io
warnings.filterwarnings('ignore')

BASE = r"C:\Users\Khurram Shahzad\Desktop\projects\khurram\khurram"

daily    = pd.read_csv(f'{BASE}\\daily_rides.csv',    sep=';')
hourly   = pd.read_csv(f'{BASE}\\hourly_rides.csv',   sep=',')
stations = pd.read_csv(f'{BASE}\\subway_stations.csv', sep=';')

print('All datasets loaded!')
print(f'  daily_rides:      {daily.shape[0]:,} rows x {daily.shape[1]} cols')
print(f'  hourly_rides:     {hourly.shape[0]:,} rows x {hourly.shape[1]} cols')
print(f'  subway_stations:  {stations.shape[0]:,} rows x {stations.shape[1]} cols')


In [ ]:
print('=' * 50)
print('DAILY RIDES COLUMNS:')
print(daily.columns.tolist())
print('\nFirst 3 rows:')
display(daily.head(3))

print('=' * 50)
print('\nHOURLY RIDES COLUMNS:')
print(hourly.columns.tolist())
print('\nFirst 3 rows:')
display(hourly.head(3))

print('=' * 50)
print('\nSUBWAY STATIONS COLUMNS:')
print(stations.columns.tolist())
print('\nFirst 3 rows:')
display(stations.head(3))


In [ ]:
print(f'Hourly rides rows: {hourly.shape[0]:,}')
print(f'\nFirst 5 rows:')
display(hourly.head())

if hourly.shape[0] == 0:
    print('\nRe reading with different settings...')
    hourly = pd.read_csv(f'{BASE}\\hourly_rides.csv', sep=';', encoding='latin-1')
    print(f'After re read: {hourly.shape[0]:,} rows')
    display(hourly.head())


In [ ]:
print('Count column dtype:', daily['Count'].dtype)
print('Sample values:')
print(daily['Count'].head(10))

daily['Count'] = pd.to_numeric(daily['Count'].astype(str).str.replace(',', ''), errors='coerce')
daily['Date']  = pd.to_datetime(daily['Date'])
daily['DayOfWeek'] = daily['Date'].dt.day_name()
daily['Month']     = daily['Date'].dt.to_period('M').astype(str)
daily['WeekNum']   = daily['Date'].dt.isocalendar().week.astype(int)
daily['IsWeekend'] = daily['Date'].dt.dayofweek >= 5

print('\nDaily rides cleaned')
print(f'   Date range: {daily["Date"].min().date()} to {daily["Date"].max().date()}')
print(f'   Modes: {daily["Mode"].unique()}')
display(daily.head())

hourly['transit_timestamp'] = pd.to_datetime(hourly['transit_timestamp'])
hourly['ridership']  = pd.to_numeric(hourly['ridership'], errors='coerce')
hourly['transfers']  = pd.to_numeric(hourly['transfers'], errors='coerce')
hourly['latitude']   = pd.to_numeric(hourly['latitude'], errors='coerce')
hourly['longitude']  = pd.to_numeric(hourly['longitude'], errors='coerce')
hourly['Hour']      = hourly['transit_timestamp'].dt.hour
hourly['DayOfWeek'] = hourly['transit_timestamp'].dt.day_name()
hourly['Date']      = hourly['transit_timestamp'].dt.date
hourly['Month']     = hourly['transit_timestamp'].dt.to_period('M').astype(str)
hourly['IsWeekend'] = hourly['transit_timestamp'].dt.dayofweek >= 5

print('\nHourly rides cleaned')
print(f'   Date range: {hourly["transit_timestamp"].min()} to {hourly["transit_timestamp"].max()}')
print(f'   Stations: {hourly["station_complex"].nunique()}')
print(f'   Boroughs: {hourly["borough"].unique()}')
display(hourly.head())

stations['GTFS Latitude']  = pd.to_numeric(stations['GTFS Latitude'], errors='coerce')
stations['GTFS Longitude'] = pd.to_numeric(stations['GTFS Longitude'], errors='coerce')
print(f'\nStations cleaned: {stations.shape[0]} stations across {stations["Borough"].nunique()} boroughs')


In [ ]:
daily_2026 = daily[daily['Date'] >= '2026-01-01'].copy()

print('DATASET SUMMARY')
print(f'\n   Daily 2026: {daily_2026.shape[0]:,} rows | {daily_2026["Date"].min().date()} to {daily_2026["Date"].max().date()}')
print(f'   Hourly:     {hourly.shape[0]:,} rows | {hourly["transit_timestamp"].min().date()} to {hourly["transit_timestamp"].max().date()}')
print(f'   Stations:   {stations.shape[0]} stations across {stations["Borough"].nunique()} boroughs')

mode_labels = {
    'Subway': 'Subway', 'Bus': 'Bus', 'LIRR': 'Long Island Rail Road',
    'MNR': 'Metro North Railroad', 'AAR': 'Access A Ride',
    'BT': 'Bridges & Tunnels', 'SIR': 'Staten Island Railway',
    'CRZ Entries': 'Congestion Zone', 'CBD Entries': 'CBD Entries'
}

avg_by_mode = daily_2026.groupby('Mode')['Count'].mean().sort_values(ascending=False)
print('\nAverage Daily Ridership by Mode (2026):')
for mode, val in avg_by_mode.items():
    print(f'   {mode_labels.get(mode, mode):.<35} {val:>12,.0f}')


In [ ]:
subway_bus = daily_2026[daily_2026['Mode'].isin(['Subway', 'Bus', 'LIRR', 'MNR', 'SIR'])]
daily_pivot = subway_bus.pivot_table(index='Date', columns='Mode', values='Count', aggfunc='sum').fillna(0)

fig, ax = plt.subplots(figsize=(14, 6))
daily_pivot.plot.area(ax=ax, colormap='Set2', alpha=0.8)
ax.set_title('Daily Public Transit Ridership in NYC (2026)', fontsize=16, fontweight='bold')
ax.set_ylabel('Daily Riders')
ax.set_xlabel('')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: format(int(x), ',')))
ax.legend(title='Transit Mode')
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'daily_transit_ridership_2026.jpg'), dpi=400, bbox_inches='tight')
plt.show()
print('Saved daily_transit_ridership_2026.jpg')


In [ ]:
wk_comparison = daily_2026.groupby(['Mode', 'IsWeekend'])['Count'].mean().reset_index()
wk_comparison['DayType'] = wk_comparison['IsWeekend'].map({True: 'Weekend', False: 'Weekday'})
wk_comparison = wk_comparison[wk_comparison['Mode'].isin(['Subway', 'Bus', 'LIRR', 'MNR'])]

fig, ax = plt.subplots(figsize=(10, 6))
modes = wk_comparison['Mode'].unique()
x = np.arange(len(modes))
width = 0.35
weekday = wk_comparison[wk_comparison['DayType']=='Weekday'].set_index('Mode')['Count'].reindex(modes)
weekend = wk_comparison[wk_comparison['DayType']=='Weekend'].set_index('Mode')['Count'].reindex(modes)
ax.bar(x - width/2, weekday, width, label='Weekday', color='#2196F3')
ax.bar(x + width/2, weekend, width, label='Weekend', color='#FF9800')
ax.set_xticks(x)
ax.set_xticklabels(modes)
ax.set_title('Weekday vs Weekend Average Ridership by Mode', fontsize=16, fontweight='bold')
ax.set_ylabel('Average Daily Riders')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: format(int(x), ',')))
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'weekday_vs_weekend_ridership.jpg'), dpi=400, bbox_inches='tight')
plt.show()
print('Saved weekday_vs_weekend_ridership.jpg')


In [ ]:
hour_day = hourly.groupby(['DayOfWeek', 'Hour'])['ridership'].sum().reset_index()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
hour_day_pivot = hour_day.pivot_table(index='DayOfWeek', columns='Hour', values='ridership')
hour_day_pivot = hour_day_pivot.reindex(day_order)

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(hour_day_pivot, cmap='YlOrRd', fmt=',.0f', ax=ax, cbar_kws={'label': 'Total Riders'})
ax.set_title('Ridership Heatmap: Hour of Day x Day of Week', fontsize=16, fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'ridership_heatmap_hour_day.jpg'), dpi=400, bbox_inches='tight')
plt.show()
print('Saved ridership_heatmap_hour_day.jpg')


In [ ]:
hourly_curve = hourly.groupby(['Hour', 'IsWeekend'])['ridership'].mean().reset_index()
hourly_curve['DayType'] = hourly_curve['IsWeekend'].map({True: 'Weekend', False: 'Weekday'})

fig, ax = plt.subplots(figsize=(12, 6))
for dtype, color in [('Weekday', '#2196F3'), ('Weekend', '#FF9800')]:
    subset = hourly_curve[hourly_curve['DayType'] == dtype]
    ax.plot(subset['Hour'], subset['ridership'], marker='o', color=color, label=dtype, linewidth=2)
ax.set_title('Average Hourly Ridership: Weekday vs Weekend', fontsize=16, fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Avg Ridership per Station Route')
ax.set_xticks(range(24))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'hourly_ridership_weekday_weekend.jpg'), dpi=400, bbox_inches='tight')
plt.show()
print('Saved hourly_ridership_weekday_weekend.jpg')


In [ ]:
top_stations = hourly.groupby('station_complex')['ridership'].sum().sort_values(ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(12, 8))
colors = plt.cm.Blues(np.linspace(0.3, 1, len(top_stations)))
ax.barh(top_stations.index, top_stations.values, color=colors)
ax.set_title('Top 20 Busiest Subway Stations (Jan 2026)', fontsize=16, fontweight='bold')
ax.set_xlabel('Total Riders')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: format(int(x), ',')))
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'top20_busiest_stations.jpg'), dpi=400, bbox_inches='tight')
plt.show()
print('Saved top20_busiest_stations.jpg')


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

borough_total = hourly.groupby('borough')['ridership'].sum()
ax1.pie(borough_total, labels=borough_total.index, autopct='%1.1f%%', colors=plt.cm.Set2.colors)
ax1.set_title('Total Ridership by Borough', fontsize=14, fontweight='bold')

borough_avg = hourly.groupby('borough')['ridership'].mean().sort_values(ascending=True)
ax2.barh(borough_avg.index, borough_avg.values, color='#2196F3')
ax2.set_title('Avg Ridership per Station', fontsize=14, fontweight='bold')
ax2.set_xlabel('Average Ridership')

plt.suptitle('Ridership by Borough', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'ridership_by_borough.jpg'), dpi=400, bbox_inches='tight')
plt.show()
print('Saved ridership_by_borough.jpg')


In [ ]:
station_geo = hourly.groupby(['station_complex', 'latitude', 'longitude']).agg(
    total_ridership=('ridership', 'sum'),
    avg_ridership=('ridership', 'mean')
).reset_index()
station_geo = station_geo.dropna(subset=['latitude', 'longitude'])

nyc_map = folium.Map(location=[40.7128, -74.0060], zoom_start=11, tiles='CartoDB positron')

heat_data = station_geo[['latitude', 'longitude', 'total_ridership']].values.tolist()
HeatMap(heat_data, radius=15, blur=10, max_zoom=13,
        gradient={0.2: 'blue', 0.4: 'lime', 0.6: 'yellow', 0.8: 'orange', 1: 'red'}
).add_to(nyc_map)

top10 = station_geo.nlargest(10, 'total_ridership')
for _, row in top10.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=8, color='red', fill=True,
        popup=f"<b>{row['station_complex']}</b><br>Riders: {row['total_ridership']:,.0f}",
        tooltip=row['station_complex']
    ).add_to(nyc_map)

print('Interactive NYC Subway Ridership Heatmap')
nyc_map.save(os.path.join(BASE, 'ridership_heatmap.html'))
print('Saved ridership_heatmap.html')
nyc_map


In [ ]:
payment = hourly.groupby('payment_method')['ridership'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 8))
colors = plt.cm.Pastel1.colors[:len(payment)]
wedges, texts, autotexts = ax.pie(payment, labels=payment.index, autopct='%1.1f%%',
                                   colors=colors, pctdistance=0.8,
                                   wedgeprops=dict(width=0.4))
ax.set_title('Ridership by Payment Method', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'ridership_by_payment_method.jpg'), dpi=400, bbox_inches='tight')
plt.show()
print('Saved ridership_by_payment_method.jpg')


In [ ]:
subway_monthly = daily[daily['Mode'] == 'Subway'].copy()
subway_monthly = subway_monthly.groupby('Month')['Count'].mean().reset_index()

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(range(len(subway_monthly)), subway_monthly['Count'], marker='o', linewidth=2, color='#2196F3')
ax.set_title('NYC Subway Average Daily Ridership: COVID Recovery (2020 to 2026)', fontsize=16, fontweight='bold')
ax.set_ylabel('Avg Daily Subway Riders')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: format(int(x), ',')))
tick_positions = range(0, len(subway_monthly), 3)
ax.set_xticks(list(tick_positions))
ax.set_xticklabels([subway_monthly['Month'].iloc[i] for i in tick_positions], rotation=45, ha='right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'subway_covid_recovery_trend.jpg'), dpi=400, bbox_inches='tight')
plt.show()
print('Saved subway_covid_recovery_trend.jpg')


In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow = daily_2026[daily_2026['Mode'] == 'Subway'].groupby('DayOfWeek')['Count'].mean().reindex(day_order)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(dow)))
ax.bar(dow.index, dow.values, color=colors)
ax.set_title('Average Subway Ridership by Day of Week (2026)', fontsize=16, fontweight='bold')
ax.set_ylabel('Average Daily Riders')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: format(int(x), ',')))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'subway_ridership_by_day_of_week.jpg'), dpi=400, bbox_inches='tight')
plt.show()
print('Saved subway_ridership_by_day_of_week.jpg')
